# Otras Tareas de Vision

Aqui vamos a ver varias tareas de vision que son muy utiles en proyectos reales. Son ejemplos rapidos para que veas como funcionan y te sirvan de base.

**Lo que veremos:**
- Estimacion de Pose (keypoints del cuerpo)
- OCR (leer texto de imagenes)
- Background Removal (quitar fondos)
- Super Resolution (mejorar calidad)
- Keypoint Matching (emparejar puntos entre imagenes)
- Depth Estimation (calcular profundidad)

Todo con modelos de Hugging Face, sin necesidad de entrenar nada.

## Configuración e Imports

In [1]:
import torch
import transformers
from transformers import pipeline
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2
import requests
from io import BytesIO
import warnings
warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch: {torch.__version__}, device: {device}")

torch: 2.7.1+cu128, device: cuda


## Estimacion de Pose

Detecta puntos clave del cuerpo humano: hombros, codos, rodillas, etc. Muy util para analisis deportivo, fisioterapia o videojuegos.

**Ejercicio**: Completa esta seccion usando VitPose o similar.

In [ ]:
# TODO: Implementar deteccion de pose
# Pistas:
# - Primero detectar personas con un detector (RT-DETR, YOLO, etc)
# - Luego aplicar modelo de pose (VitPose) a cada persona detectada
# - Dibujar el skeleton conectando los keypoints

pass

## OCR - Leer Texto de Imagenes

Extraer texto de imagenes sirve para digitalizar documentos, leer matriculas, procesar facturas, etc.

**Libreria**: `easyocr` - Muy facil de usar, soporta muchos idiomas y funciona bien con texto en cualquier posicion de la imagen.

In [2]:
try:
    import easyocr
except:
    %pip install easyocr
    import easyocr

# Cargar reader (la primera vez descarga el modelo)
# Idiomas: 'en' ingles, 'es' español, etc
reader = easyocr.Reader(['en', 'es'], gpu=torch.cuda.is_available())
print("EasyOCR cargado")

  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached shapely-2.1.2-cp312-cp312-win_amd64.whl.metadata (7.1 kB)
  Using cached pyclipper-1.3.0.post6-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
Using cached easyocr-1.7.2-py3-none-any.whl (2.9 MB)
Using cached pyclipper-1.3.0.post6-cp312-cp312-win_amd64.whl (110 kB)
Using cached shapely-2.1.2-cp312-cp312-win_amd64.whl (1.7 MB)

   -------------------- ------------------- 2/4 [Shapely]
   -------------------- ------------------- 2/4 [Shapely]
   ------------------------------ --------- 3/4 [easyocr]
   ------------------------------ --------- 3/4 [easyocr]
   ---------------------------------------- 4/4 [easyocr]

Note: you may need to restart the kernel to use updated packages.
Progress: |██████████████████████████████████████████████████| 100.0% CompleteEasyOCR cargado


In [ ]:
# Imagen de ejemplo con texto
url_cars = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/cars.jpg"
image_cars = Image.open(BytesIO(requests.get(url_cars).content)).convert("RGB")

# Leer texto
results = reader.readtext(np.array(image_ocr))

# Dibujar bounding boxes y texto
img_draw = np.array(image_ocr).copy()
for (bbox, text, score) in results:
    # bbox tiene 4 puntos, tomamos esquinas
    pts = np.array(bbox).astype(int)
    cv2.polylines(img_draw, [pts], True, (0, 255, 0), 2)
    cv2.putText(img_draw, text, (pts[0][0], pts[0][1]-5), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

# Mostrar
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(image_ocr)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(img_draw)
axes[1].set_title("Texto detectado")
axes[1].axis("off")
plt.tight_layout()
plt.show()

# Imprimir texto encontrado
print("Texto encontrado:")
for (bbox, text, score) in results:
    print(f"  '{text}' (confianza: {score:.2f})")

SSLError: HTTPSConnectionPool(host='fki.tic.heia-fr.ch', port=443): Max retries exceeded with url: /static/img/a01-122-02-00.jpg (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1010)')))

## Background Removal - Quitar Fondos

Sirve para aislar objetos del fondo. Muy usado en ecommerce para catalogos de productos, edicion de fotos, etc.

**Modelo**: `briaai/RMBG-1.4` - Un modelo especializado en eliminar fondos que funciona muy bien con personas y objetos.

In [ ]:
# Cargar modelo
bg_pipe = pipeline(
    task="image-segmentation",
    model="briaai/RMBG-1.4",
    trust_remote_code=True
)
print("Modelo Background Removal cargado")

In [ ]:
# Cargar imagen de ejemplo
url = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/dog2.jpg"
image_bg = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

# Quitar fondo
result = bg_pipe(image_bg)

# Mostrar resultado
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image_bg)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(result)
axes[1].set_title("Sin fondo")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Super Resolution - Mejorar Calidad de Imagenes

Aumenta la resolucion de imagenes pixeladas o de baja calidad. Util para restaurar fotos antiguas, mejorar capturas de video, etc.

**Modelo**: `caidas/swin2SR-classical-sr-x4-64` - Aumenta la resolucion x4 usando Swin Transformer.

In [ ]:
# Usamos pipeline que es mas sencillo
sr_pipe = pipeline(
    task="image-to-image",
    model="caidas/swin2SR-classical-sr-x4-64"
)
print("Modelo Super Resolution cargado")

In [ ]:
# Cargar imagen y reducir resolucion para simular imagen de baja calidad
url = "https://github.com/sergiovillanueva/modelos_fundacionales/raw/main/assets/dog.jpg"
image_sr = Image.open(BytesIO(requests.get(url).content)).convert("RGB")

# Reducir a 64x64 para que se note el efecto
image_low = image_sr.resize((64, 64), Image.BILINEAR)

# Aplicar super resolution (x4 = 256x256)
image_high = sr_pipe(image_low)

# Mostrar comparacion
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_sr)
axes[0].set_title(f"Original {image_sr.size}")
axes[0].axis("off")

axes[1].imshow(image_low)
axes[1].set_title(f"Reducida {image_low.size}")
axes[1].axis("off")

axes[2].imshow(image_high)
axes[2].set_title(f"Super Resolution {image_high.size}")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## Keypoint Matching - Emparejar Puntos entre Imagenes

Encuentra correspondencias entre puntos de dos imagenes de la misma escena. Se usa mucho en:
- Reconstruccion 3D (Structure from Motion)
- Localizacion visual (donde estoy?)
- Stitching de panoramas

**Modelo**: `ETH-CVG/lightglue_superpoint` - Combina SuperPoint (detector de keypoints) con LightGlue (matcher rapido).

In [ ]:
from transformers import AutoImageProcessor, AutoModel

# Cargar modelo
match_processor = AutoImageProcessor.from_pretrained("ETH-CVG/lightglue_superpoint")
match_model = AutoModel.from_pretrained("ETH-CVG/lightglue_superpoint")
print("Modelo Keypoint Matching cargado")

In [ ]:
# Cargar dos imagenes de la misma escena (Capitolio de EEUU desde diferentes angulos)
url1 = "https://raw.githubusercontent.com/magicleap/SuperGluePretrainedNetwork/refs/heads/master/assets/phototourism_sample_images/united_states_capitol_98169888_3347710852.jpg"
url2 = "https://raw.githubusercontent.com/magicleap/SuperGluePretrainedNetwork/refs/heads/master/assets/phototourism_sample_images/united_states_capitol_26757027_6717084061.jpg"

image1 = Image.open(requests.get(url1, stream=True).raw)
image2 = Image.open(requests.get(url2, stream=True).raw)
images = [image1, image2]

# Procesar
inputs = match_processor(images, return_tensors="pt")
with torch.no_grad():
    outputs = match_model(**inputs)

# Post-procesar para obtener matches
image_sizes = [[(img.height, img.width) for img in images]]
matches = match_processor.post_process_keypoint_matching(outputs, image_sizes, threshold=0.2)

# Extraer keypoints y scores
kpts0 = matches[0]["keypoints0"].numpy()
kpts1 = matches[0]["keypoints1"].numpy()
scores = matches[0]["matching_scores"].numpy()
n_matches = len(kpts0)
print(f"Matches encontrados: {n_matches}")

# Crear imagen concatenada para visualizar
img1_np = np.array(image1)
img2_np = np.array(image2)

# Redimensionar a la misma altura
h1, w1 = img1_np.shape[:2]
h2, w2 = img2_np.shape[:2]
h = min(h1, h2)
img1_resized = cv2.resize(img1_np, (int(w1 * h / h1), h))
img2_resized = cv2.resize(img2_np, (int(w2 * h / h2), h))

# Concatenar
combined = np.hstack([img1_resized, img2_resized])
offset = img1_resized.shape[1]  # Offset para los puntos de la segunda imagen

# Dibujar matches (solo los mejores 50 para no saturar)
n_draw = min(50, n_matches)
indices = np.argsort(scores)[::-1][:n_draw]  # Los de mayor score

for idx in indices:
    # Escalar puntos al tamaño redimensionado
    pt1 = (int(kpts0[idx][0] * h / h1), int(kpts0[idx][1] * h / h1))
    pt2 = (int(kpts1[idx][0] * h / h2) + offset, int(kpts1[idx][1] * h / h2))
    
    # Color segun score (verde = bueno, rojo = malo)
    color = (0, int(255 * scores[idx]), int(255 * (1 - scores[idx])))
    
    cv2.line(combined, pt1, pt2, color, 1)
    cv2.circle(combined, pt1, 3, (255, 0, 0), -1)
    cv2.circle(combined, pt2, 3, (255, 0, 0), -1)

# Mostrar
plt.figure(figsize=(16, 8))
plt.imshow(combined)
plt.title(f"Keypoint Matching: {n_matches} correspondencias (mostrando top {n_draw})")
plt.axis("off")
plt.show()

## Depth Estimation - Calcular Profundidad

Estima la distancia de cada pixel a la camara. Se usa para:
- Navegacion de robots y coches autonomos
- Realidad aumentada (poner objetos 3D en escenas reales)
- Efectos de desenfoque tipo retrato

**Modelo**: `depth-anything/Depth-Anything-V2-Small-hf` - Modelo SOTA que funciona con cualquier imagen.

In [ ]:
# Cargar modelo
depth_pipe = pipeline(
    task="depth-estimation",
    model="depth-anything/Depth-Anything-V2-Small-hf"
)
print("Modelo Depth Estimation cargado")

In [ ]:
# Imagen de ejemplo
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image_depth = Image.open(requests.get(url, stream=True).raw)

# Estimar profundidad
result = depth_pipe(image_depth)
depth_map = result["depth"]

# Mostrar resultado
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image_depth)
axes[0].set_title("Imagen original")
axes[0].axis("off")

# El mapa de profundidad: colores claros = cerca, oscuros = lejos
axes[1].imshow(depth_map, cmap="plasma")
axes[1].set_title("Mapa de profundidad")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Resumen

En este notebook hemos visto varias tareas de vision:

- **Pose**: Detectar keypoints del cuerpo (ejercicio pendiente)
- **OCR**: Leer texto con TrOCR
- **Background Removal**: Quitar fondos con RMBG
- **Super Resolution**: Mejorar calidad x4 con Swin2SR
- **Keypoint Matching**: Emparejar puntos entre imagenes con LightGlue
- **Depth Estimation**: Calcular profundidad con Depth Anything V2

Todos estos modelos estan disponibles en Hugging Face y se pueden usar facilmente con `pipeline()` o cargando el modelo directamente.

Para explorar mas tareas: https://huggingface.co/tasks